# Single-sample token graph visualization

One point/node is one token. This notebook directly uses `ResearchDataset` through `SampleGraphVisualizer` and needs only the reconstructed canonical test split. `visualize()` generates the full token graph, an error-centered incoming-edge heatmap, node-level t-SNE, graph-state trajectories, and a matched correct-control comparison.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from research_analysis import SampleGraphVisualizer

DATA_ROOT = Path(
    "/share/home/tm902089733300000/a903202310/lys/data/RAGTruth/"
    "model_traces/llama31_8b/test"
)
OUTPUT_ROOT = Path(
    "/share/home/tm902089733300000/a903202310/lys/data/RAGTruth/"
    "visualizations/llama31_8b/single_sample_graphs"
)

ERROR_INDEX = 0          # choose another hallucinated sample by index
ERROR_SAMPLE_ID = None   # or set an explicit sample ID string, e.g. "10005"
NODE_TSNE_MODE = "structure"  # "structure" = 12-D graph state; "attention" = attention-diagonal node features
LOCAL_RADIUS = 12


In [ ]:
viewer = SampleGraphVisualizer(DATA_ROOT)
print(f"test samples: {len(viewer.dataset)}")
print(f"hallucinated samples: {len(viewer.error_sample_ids)}")
print(f"fully correct samples: {len(viewer.correct_sample_ids)}")
viewer.list_errors(limit=10)


In [ ]:
sample_id = (
    str(ERROR_SAMPLE_ID)
    if ERROR_SAMPLE_ID is not None
    else viewer.error_sample_ids[ERROR_INDEX]
)
print("selected error sample:", sample_id)
print("positive runs:", viewer.labels.positive_runs(sample_id))
viewer.dataset[sample_id].metadata


## One-call visualization

The token graph keeps the original token order and relation edges. The heatmap zooms around the first hallucination onset. In the node-level t-SNE, one point is one response token in `structure` mode; the sequential line shows the generation trajectory through the 12-D graph-state space.


In [ ]:
result = viewer.visualize(
    sample_id,
    output_dir=OUTPUT_ROOT / sample_id,
    local_radius=LOCAL_RADIUS,
    node_tsne_mode=NODE_TSNE_MODE,
    include_control=True,
)
print("matched correct control:", result["control_sample_id"])
print("saved to:", OUTPUT_ROOT / sample_id)


## Optional: inspect the raw graph view

`graph_view()` is the data-layer object used by every visualization: token IDs, unique source→target relations, 12-D response-node states, and token labels are already aligned.


In [ ]:
view = viewer.dataset[sample_id].graph_view(viewer.labels)
print("tokens:", view["num_tokens"], "response tokens:", view["num_response_tokens"])
print("unique relations:", len(view["relations"]["weight"]))
print("node-state matrix:", tuple(view["response_features"].shape))
print("feature names:", view["structural_feature_names"])


## Optional: switch node embedding

`structure` tests whether hallucination tokens occupy a different local graph-state region. `attention` instead projects the attention-diagonal feature vector of every prompt/response token.


In [ ]:
# Example:
# viewer.plot_node_tsne(sample_id, mode="attention")
# viewer.plot_edge_heatmap(sample_id, center=view["positive_runs"][0][0], radius=20)
